In [1]:
import pandas as pd
# reads csv files into dataframes
df0 = pd.read_csv("CRMLSSold202506.csv")
df1 = pd.read_csv("CRMLSSold202507.csv")
df2 = pd.read_csv("CRMLSSold202508.csv")
df3 = pd.read_csv("CRMLSSold202509.csv")
df4 = pd.read_csv("CRMLSSold202510.csv")
df5 = pd.read_csv("CRMLSSold202511.csv")
df6 = pd.read_csv("CRMLSSold202512.csv")
# Creates train test split
df_train = pd.concat([df0, df1, df2, df3, df4, df5], axis = 0)
df_test = df6
# Narrows down the data
df_train = df_train[(df_train["PropertyType"] == "Residential") & (df_train["PropertySubType"] == "SingleFamilyResidence")]
df_test = df_test[(df_test["PropertyType"] == "Residential") & (df_test["PropertySubType"] == "SingleFamilyResidence")]
# df_to_use.columns
# df_to_use.shape

# filters out invalid data
df_train = df_train[(df_train["ClosePrice"] > 0) & (df_train["LivingArea"] > 0)]
df_train = df_train.dropna(subset = ["Latitude", "Longitude"])

df_test = df_test[(df_test["ClosePrice"] > 0) & (df_test["LivingArea"] > 0)]
df_test = df_test.dropna(subset = ["Latitude", "Longitude"])



C:\Users\jehil\AppData\Local\Temp\ipykernel_44680\2529616134.py:3: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df0 = pd.read_csv("CRMLSSold202506.csv")


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import BallTree

def add_spatial_features(df, k=6):
    """
    Step 3: Feature Engineering (Spatial Data Science).
    
    Standard regression models fail to capture "Location".
    My solution: Create a 'Spatial Lag' feature.
    
    Logic: I use a BallTree (KNN) to find the nearest 5 neighbors for every house 
    and calculate their average price. This quantifies the 'neighborhood value'.
    """
    print(">> Engineering SpatialLag_Price (KNN algorithm)...")
    
    # Haversine metric requires radians
    df['lat_rad'] = np.radians(df['Latitude'])
    df['lon_rad'] = np.radians(df['Longitude'])
    
    # Build the tree for fast spatial indexing
    tree = BallTree(df[['lat_rad', 'lon_rad']].values, metric='haversine')
    
    # Query k nearest neighbors (1st one is the point itself, so we take k=6)
    dist, ind = tree.query(df[['lat_rad', 'lon_rad']].values, k=k)
    
    # Calculate mean price of neighbors (excluding the house itself)
    prices = df['ClosePrice'].values
    neighbor_indices = ind[:, 1:] 
    
    neighbor_prices = prices[neighbor_indices]
    df['SpatialLag_Price'] = np.mean(neighbor_prices, axis=1)
    
    return df

In [3]:
# add feature engineering to both train and test
df_train = add_spatial_features(df_train)
df_test = add_spatial_features(df_test)

>> Engineering SpatialLag_Price (KNN algorithm)...
>> Engineering SpatialLag_Price (KNN algorithm)...


In [4]:
df_train

,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,...,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,lat_rad,lon_rad,SpatialLag_Price
3,TheInlandGateway,TheInlandGateway,NaN,True,NaN,NaN,False,889000.0,523319952,hutton@cbappteam.com,...,True,2.0,Rim of the World,92352,0.0,9600.0,NaN,0.598032,-2.045893,765300.0
10,BeverlyHillsGreaterLa,BeverlyHillsGreaterLa,Wood,True,NaN,NaN,False,1899999.0,1118606385,chase.campen@compass.com,...,False,NaN,NaN,90046,NaN,10400.0,NaN,0.595297,-2.066301,4998000.0
11,Mlslistings,Mlslistings,NaN,False,NaN,NaN,NaN,NaN,1118606192,stanleylo@greenbanker.com,...,False,3.0,Other,94010,NaN,22505.0,NaN,0.655675,-2.136078,5073000.0
13,PacificWest,PacificWest,NaN,True,NaN,NaN,False,865000.0,1118604114,matt@majorleaguesocal.com,...,False,2.0,Placentia-Yorba Linda Unified,92886,0.0,4800.0,NaN,0.591772,-2.055610,3840000.0
14,BayEast,BayEast,"Carpet,Laminate",NaN,NaN,NaN,False,875000.0,1118603794,brianrowland.homes@gmail.com,...,False,4.0,NaN,94546,NaN,5500.0,NaN,0.658092,-2.130339,1109200.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19057,CoastalMendocino,CoastalMendocino,"Laminate,Tile",True,NaN,NaN,False,830000.0,1048537311,linda@mendosir.com,...,NaN,2.0,NaN,95437,NaN,79714.8,NaN,0.688545,-2.160030,588000.0
19060,PacificWest,PacificWest,NaN,False,NaN,NaN,False,1040000.0,1044835979,Macias4inc@gmail.com,...,False,2.0,Los Angeles Unified,90031,0.0,7219.0,NaN,0.594822,-2.063033,783000.0
19071,CitrusValley,CitrusValley,NaN,True,NaN,NaN,False,820000.0,1034167333,jenny@primetimerealtyca.com,...,False,2.0,San Diego Unified,92114,0.0,6900.0,NaN,0.570786,-2.043317,1219000.0
19077,HighDesert,HighDesert,NaN,True,NaN,NaN,False,295000.0,1031766878,oliver@revilorealty.com,...,False,0.0,Antelope Valley Union,93544,0.0,218781.0,NaN,0.602170,-2.055562,280400.0


In [5]:

cols = ["ClosePrice", "BedroomsTotal", "BathroomsTotalInteger", "LivingArea", "LotSizeSquareFeet", "YearBuilt", "Latitude", "Longitude", "SpatialLag_Price"]
df_filtered_train = df_train[cols]
df_test_filtered = df_test[cols]

In [6]:
#finds number of na values for each feature
df_filtered_train.isna().sum()
df_test_filtered.isna().sum()

ClosePrice                 0
BedroomsTotal              0
BathroomsTotalInteger      0
LivingArea                 0
LotSizeSquareFeet        156
YearBuilt                  6
Latitude                   0
Longitude                  0
SpatialLag_Price           0
dtype: int64

In [7]:
df_filtered_train["LotSizeSquareFeet"][df_filtered_train["LotSizeSquareFeet"] == 0] = None
df_filtered_train = df_filtered_train.dropna(subset = ["YearBuilt", "BathroomsTotalInteger", "LotSizeSquareFeet"])

df_test_filtered["LotSizeSquareFeet"][df_test_filtered["LotSizeSquareFeet"] == 0] = None
df_test_filtered = df_test_filtered.dropna(subset = ["YearBuilt", "BathroomsTotalInteger", "LotSizeSquareFeet"])
df_test_filtered

C:\Users\jehil\AppData\Local\Temp\ipykernel_44680\834665494.py:1: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_filtered_train["LotSizeSquareFeet"][df_filtered_train["LotSizeSquareFeet"] == 0] = None
C:\Users\jehil\AppData\Local\Temp\ipyk

,ClosePrice,BedroomsTotal,BathroomsTotalInteger,LivingArea,LotSizeSquareFeet,YearBuilt,Latitude,Longitude,SpatialLag_Price
0,1998000.0,4.0,2.0,2045.0,10080.0,1968.0,37.871927,-122.029871,1617600.0
2,2214421.0,4.0,4.0,3050.0,34745.0,1957.0,34.150680,-118.580650,2193600.0
3,1200000.0,4.0,2.0,1594.0,6600.0,1978.0,37.306405,-121.835428,1173600.0
7,3100000.0,5.0,3.0,2700.0,8262.0,2025.0,37.272636,-121.921351,2607000.0
9,2900000.0,5.0,4.0,2948.0,9222.0,2023.0,37.303942,-121.935424,3412800.0
...,...,...,...,...,...,...,...,...,...
20494,640000.0,3.0,3.0,2474.0,105281.0,1992.0,34.608952,-118.259648,697000.0
20496,12000000.0,5.0,4.0,5353.0,15118.0,2004.0,37.443497,-122.150296,6022000.0
20510,600000.0,3.0,2.0,2038.0,871200.0,2004.0,35.903719,-120.982965,735800.0
20511,380000.0,3.0,2.0,1452.0,9620.0,1971.0,34.261375,-117.211842,851000.0


In [8]:

df_test_filtered.isna().sum()

ClosePrice               0
BedroomsTotal            0
BathroomsTotalInteger    0
LivingArea               0
LotSizeSquareFeet        0
YearBuilt                0
Latitude                 0
Longitude                0
SpatialLag_Price         0
dtype: int64

In [9]:
# df_filtered.isna().sum()
# 5108/df_filtered.shape[0]

In [10]:
# Encodes new construction variable into binary variable
# df_filtered_train["NewConstructionYN"][df_filtered_train["NewConstructionYN"] == False] = 0
# df_filtered_train["NewConstructionYN"][df_filtered_train["NewConstructionYN"] == True] = 1
# df_filtered_train["NewConstructionYN"].value_counts()

# df_test_filtered["NewConstructionYN"][df_test_filtered["NewConstructionYN"] == False] = 0
# df_test_filtered["NewConstructionYN"][df_test_filtered["NewConstructionYN"] == True] = 1
# df_test_filtered["NewConstructionYN"].value_counts()

In [13]:
#scales numerical variables only in training set
from sklearn.preprocessing import StandardScaler
df_filtered_train.dtypes
num_cols = ["YearBuilt", "BathroomsTotalInteger", "BedroomsTotal", "LotSizeSquareFeet"]
scaler = StandardScaler()
df_filtered_train[num_cols] = scaler.fit_transform(df_filtered_train[num_cols])
df_filtered_train
df_test_filtered[num_cols] = scaler.transform(df_test_filtered[num_cols])

In [14]:
#remove bottom 0.5% and top 0.5% of ClosePrice values in the testing set
c_price = "ClosePrice"


In [15]:
#test set done separately
test_low = df_test_filtered[c_price].quantile(0.005)
test_high = df_test_filtered[c_price].quantile(0.995)
df_test_cleaned = df_test_filtered[(df_test_filtered[c_price] >= test_low) & (df_test_filtered[c_price] <= test_high)]
df_test_cleaned



,ClosePrice,BedroomsTotal,BathroomsTotalInteger,LivingArea,LotSizeSquareFeet,YearBuilt,Latitude,Longitude,SpatialLag_Price
0,1998000.0,4.0,2.0,2045.0,10080.0,1968.0,37.871927,-122.029871,1617600.0
2,2214421.0,4.0,4.0,3050.0,34745.0,1957.0,34.150680,-118.580650,2193600.0
3,1200000.0,4.0,2.0,1594.0,6600.0,1978.0,37.306405,-121.835428,1173600.0
7,3100000.0,5.0,3.0,2700.0,8262.0,2025.0,37.272636,-121.921351,2607000.0
9,2900000.0,5.0,4.0,2948.0,9222.0,2023.0,37.303942,-121.935424,3412800.0
...,...,...,...,...,...,...,...,...,...
20490,479500.0,2.0,2.0,3000.0,1745449.0,1990.0,36.351756,-121.582590,1091000.0
20494,640000.0,3.0,3.0,2474.0,105281.0,1992.0,34.608952,-118.259648,697000.0
20510,600000.0,3.0,2.0,2038.0,871200.0,2004.0,35.903719,-120.982965,735800.0
20511,380000.0,3.0,2.0,1452.0,9620.0,1971.0,34.261375,-117.211842,851000.0


In [16]:
# linear regression
import statsmodels.formula.api as smf
from sklearn.metrics import r2_score


# model = smf.ols(formula = "ClosePrice ~  YearBuilt + BathroomsTotalInteger + BedroomsTotal + NewConstructionYN + LotSizeSquareFeet + Latitude + Longitude + SpatialLag_Price", data = df_filtered_train)

model = smf.ols(formula = "ClosePrice ~ BedroomsTotal + BathroomsTotalInteger + LivingArea + LotSizeSquareFeet + YearBuilt + Latitude + Longitude + SpatialLag_Price", data = df_filtered_train).fit()
print(model.summary())

# print(df_filtered_train.columns)
# print(df_test_cleaned.columns)

# df_filtered_train
predictions = model.predict(df_test_cleaned)
actual = df_test_cleaned["ClosePrice"]
# predictions = model.predict(df_test_cleaned[["PropertySubType", "YearBuilt", "BathroomsTotalInteger", "BedroomsTotal", "NewConstructionYN", "LotSizeSquareFeet"]])
# actual = df_test_cleaned["ClosePrice"]
r_squared = r2_score(actual, predictions)
r_squared

                            OLS Regression Results                            
Dep. Variable:             ClosePrice   R-squared:                       0.015
Model:                            OLS   Adj. R-squared:                  0.015
Method:                 Least Squares   F-statistic:                     126.4
Date:                Fri, 13 Feb 2026   Prob (F-statistic):          2.76e-211
Time:                        17:05:31   Log-Likelihood:            -1.1715e+06
No. Observations:               67186   AIC:                         2.343e+06
Df Residuals:                   67177   BIC:                         2.343e+06
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept             -1.992e+

-13238397098.321207

In [17]:
#random forest Rsquared
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42)
x_train = df_filtered_train[["BedroomsTotal","BathroomsTotalInteger", "LivingArea", "LotSizeSquareFeet", "YearBuilt", "Latitude", "Longitude"]]
print(type(x_train))
y_train = df_filtered_train["ClosePrice"]
rf.fit(x_train, y_train)
x_test = df_test_cleaned[["BedroomsTotal","BathroomsTotalInteger", "LivingArea", "LotSizeSquareFeet", "YearBuilt", "Latitude", "Longitude"]]
y_pred = rf.predict(x_test)
r2 = r2_score(actual, y_pred)
r2


<class 'pandas.core.frame.DataFrame'>


-101.75822017297473